In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from Quantization.temporary_functions import *
import torch
import torch.nn as nn
from collections import OrderedDict
import math
from transformers.activations import ACT2FN
from typing import Optional, Tuple, List
import torch.nn.functional as F
import copy
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding,apply_rotary_pos_emb,LlamaRMSNorm,repeat_kv
from transformers.models.llama.configuration_llama import LlamaConfig
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers.models.llama.modeling_llama import (
    apply_rotary_pos_emb,
    repeat_kv
)



In [3]:
from typing import Union
import tqdm
import numpy as np
import pdb
import math

CLIPMIN = 1e-5



def round_ste(x: torch.Tensor):
    """
    Implement Straight-Through Estimator for rounding operation.
    """
    y=x.round()
    return x+(y - x).detach()



class UniformAffineQuantizer(nn.Module):
    def __init__(
        self,
        n_bits: int = 8,
        symmetric: bool = False,
        per_channel_axes=[],
        metric="minmax",
        dynamic=False,
        dynamic_method="per_cluster",
        group_size=None,
        shape=None,
        lwc=False
    ):
        """
        support cluster quantize
        dynamic_method support per_token and per_cluster
        """
        super().__init__()
        self.symmetric = symmetric
        assert 2 <= n_bits <= 16, "bitwidth not supported"
        self.n_bits = n_bits
        self.qmin = 0
        self.qmax = 2 ** (n_bits) - 1
        self.per_channel_axes = per_channel_axes
        self.metric = metric
        self.cluster_counts = None
        self.cluster_dim = None

        self.scale = None
        self.zero_point = None
        self.round_zero_point = None

        self.cached_xmin = None
        self.cached_xmax = None
        self.dynamic = dynamic
        self.dynamic_method = dynamic_method
        self.deficiency = 0
        self.lwc = lwc

        init_value = 4.             # inti value of learnable weight clipping
        if lwc:
            if group_size:
                dim1 = int(shape[0]*math.ceil(shape[1]/group_size))
                self.deficiency = shape[-1]%group_size
                if self.deficiency > 0:
                    self.deficiency = group_size - self.deficiency
                    assert self.symmetric   # support for mlc-llm symmetric quantization
            else:
                dim1 = shape[0]
            self.upbound_factor = nn.Parameter(torch.ones((dim1,1))*init_value)
            self.lowbound_factor = nn.Parameter(torch.ones((dim1,1))*init_value)

        self.sigmoid = nn.Sigmoid()

        self.enable = True
        self.group_size = group_size

    def change_n_bits(self, n_bits):
        self.n_bits = n_bits
        self.qmin = 0
        self.qmax = 2 ** (n_bits) - 1

    def fake_quant(self, x, scale, round_zero_point):
        if self.deficiency > 0:
            pad_zeros = torch.zeros((x.shape[0],self.deficiency),dtype=x.dtype,device=x.device)
            x = torch.cat((x,pad_zeros),dim=1)

        if self.group_size:
            assert len(x.shape)==2, "only support linear layer now"
            dim1, dim2 = x.shape
            x = x.reshape(-1, self.group_size)
        x_int = round_ste(x / scale)
        if round_zero_point is not None:
            x_int = x_int.add(round_zero_point)
        x_int = x_int.clamp(self.qmin, self.qmax)
        x_dequant = x_int
        if round_zero_point is not None:
            x_dequant = x_dequant.sub(round_zero_point)
        x_dequant = x_dequant.mul(scale)
        if self.group_size:
            x_dequant = x_dequant.reshape(dim1, dim2)
        if self.deficiency > 0:
            x_dequant = x_dequant[:,:-self.deficiency]
        return x_dequant


    def forward(self, x: torch.Tensor, rotate=1):
        if self.n_bits >= 16 or not self.enable:
            return x
        if self.metric == "fix0to1":
            return x.mul_(2**self.n_bits-1).round_().div_(2**self.n_bits-1)

        if self.dynamic_method == "per_token" or self.dynamic_method == "per_channel":
            self.per_token_dynamic_calibration(x)
        else:
            raise NotImplementedError()

        x_dequant = self.fake_quant(x, self.scale, self.round_zero_point)
        return rotate * x_dequant

    def per_token_dynamic_calibration(self, x):
        if self.group_size:
            if self.deficiency == 0:
                x = x.reshape(-1,self.group_size)
            else:
                pad_zeros = torch.zeros((x.shape[0],self.deficiency),dtype=x.dtype,device=x.device)
                x = torch.cat((x,pad_zeros),dim=1)
                x = x.reshape(-1,self.group_size)
        reduce_shape = [-1]
        xmin = x.amin(reduce_shape, keepdim=True)
        xmax =  x.amax(reduce_shape, keepdim=True)
        if self.lwc:
            xmax = self.sigmoid(self.upbound_factor)*xmax
            xmin = self.sigmoid(self.lowbound_factor)*xmin
        if self.symmetric:
            abs_max = torch.max(xmax.abs(),xmin.abs())
            scale = abs_max / (2**(self.n_bits-1)-1)
            self.scale = scale.clamp(min=CLIPMIN, max=1e4)
            zero_point = (2**(self.n_bits-1)-1)*torch.ones_like(self.scale)
        else:
            range = xmax - xmin
            scale = range / (2**self.n_bits-1)
            self.scale = scale.clamp(min=CLIPMIN, max=1e4)
            zero_point = -(xmin) / (self.scale)
        self.round_zero_point = zero_point.clamp(min=-1e4, max=1e4).round()

    def register_scales_and_zeros(self):
        self.register_buffer('scales', self.scale)
        self.register_buffer('zeros', self.round_zero_point)
        del self.scale
        del self.round_zero_point


In [4]:
class QuantLinear(nn.Module):
    # def __init__(self, linear: nn.Linear, n_bits=8):
    #     super().__init__()
    #     self.register_buffer('weight', linear.weight)
    #     self.register_buffer('bias', linear.bias) if linear.bias is not None else setattr(self, 'bias', None)
    #     self.n_bits = n_bits
    # def custom_scale_formula(self, x):
    #     # Your Formula: scale = act / log2(2 + act)
    #     # We use the max absolute value as the 'act' representative for the tensor
    #     act_val = x.abs().max()
    #     # Adding 1e-5 to prevent division by zero or log of 0
    #     scale = act_val / torch.log2(2.0 + act_val).clamp(min=1e-5)
    #     return scale
    # def fake_quant(self, x):
    #     qmin, qmax = -2**(self.n_bits - 1), 2**self.n_bits - 1
    #     scale = (x.max() - x.min()) / (qmax - qmin + 1e-5)
    #     #scale=self.custom_scale_formula(x)
    #     zero_point = -x.min() / (scale + 1e-5)
    #     x_int = torch.clamp((x / scale + zero_point).round(), qmin, qmax)
    #     return (x_int - zero_point) * scale

    # def forward(self, x):
    #     # We quantize weights and activations here
    #     weight = self.fake_quant(self.weight)
    #     out = F.linear(x, weight, self.bias)
    #     return self.fake_quant(out)
    def __init__(
        self,
        org_module: nn.Linear,
        weight_quant_params: dict = {},
        act_quant_params: dict = {},
        disable_input_quant=False,
    ):
        super().__init__()
        self.fwd_kwargs = dict()
        self.fwd_func = F.linear
        self.register_buffer('weight',org_module.weight)
        if org_module.bias is not None:
            self.register_buffer('bias',org_module.bias)
        else:
            self.bias = None
        self.in_features = org_module.in_features
        self.out_features = org_module.out_features
        # de-activate the quantized forward default
        self.use_weight_quant = False
        self.use_act_quant = False
        # initialize quantizer
        self.weight_quantizer = UniformAffineQuantizer(**weight_quant_params,shape=org_module.weight.shape)
        if not disable_input_quant:
            self.act_quantizer = UniformAffineQuantizer(**act_quant_params)
        else:
            self.act_quantizer = None

        self.disable_input_quant = disable_input_quant
        self.use_temporary_parameter = False

    def forward(self, input: torch.Tensor):
        if self.use_temporary_parameter:
            weight = self.temp_weight
            bias = self.temp_bias
        elif self.use_weight_quant:
            weight = self.weight_quantizer(self.weight)
            bias = self.bias
        else:
            weight = self.weight
            bias = self.bias

        if self.use_act_quant and not self.disable_input_quant:
            input = self.act_quantizer(input)

        out = self.fwd_func(input, weight, bias, **self.fwd_kwargs)


        return out

    def set_quant_state(self, weight_quant: bool = False, act_quant: bool = False):
        self.use_weight_quant = weight_quant
        self.use_act_quant = act_quant

In [5]:
class QuantMatMul(nn.Module):
    def __init__(
        self,
        x1_quant_params: dict = {},
        x2_quant_params: dict = {},
        disable_act_quant=False,
        matmul_func=torch.bmm,
    ):
        super().__init__()
        # de-activate the quantized forward default
        self.use_act_quant = False
        # initialize quantizer
        self.i_cluster_counts = None
        self.x1_quantizer = UniformAffineQuantizer(**x1_quant_params)
        self.x2_quantizer = UniformAffineQuantizer(**x2_quant_params)
        self.matmul_func = matmul_func

        self.disable_act_quant = disable_act_quant


    def set_quant_state(self, weight_quant: bool = False, act_quant: bool = False):
        self.use_weight_quant = weight_quant
        self.use_act_quant = act_quant

    def quant_x1(self, x1):
        if self.use_act_quant:
            x1 = self.x1_quantizer(x1)
        return x1

    def quant_x2(self, x2):
        if self.use_act_quant:
            x2 = self.x2_quantizer(x2)
        return x2

    def forward(self, x1, x2):
        out = self.matmul_func(x1, x2)
        return out

In [6]:
import torch
import torch.nn as nn


'''
Modify normalization layer to adapt the training of learnable equivalent transformation
'''



class QuantLayerNorm(nn.Module):
    def __init__(self, ori_layer_norm) -> None:
        super().__init__()
        self.use_act_quant = True
        self.register_buffer('weight',ori_layer_norm.weight)
        if ori_layer_norm.bias is not None:
            self.register_buffer('bias',ori_layer_norm.bias)
        else:
            self.bias = None
        self.eps = ori_layer_norm.eps
        self.norm_func = nn.functional.layer_norm
        self.normalized_shape = ori_layer_norm.normalized_shape
        self.use_temporary_parameter = False


    def forward(self, x):
        if self.use_temporary_parameter:
            weight = self.temp_weight
            bias = self.temp_bias
        else:
            weight = self.weight
            bias = self.bias
        out = self.norm_func(x,self.normalized_shape,weight, bias,eps=self.eps)
        return out

    def set_quant_state(self, use_weight_quant, use_act_quant):
        self.use_act_quant = use_act_quant


class QuantLlamaRMSNorm(nn.Module):
    def __init__(self, ori_norm, eps=1e-6):
        """
        LlamaRMSNorm is equivalent to T5LayerNorm
        """
        super().__init__()
        self.register_buffer('weight',ori_norm.weight)
        self.bias = None
        self.variance_epsilon = eps
        self.use_temporary_parameter = False


    def forward(self, hidden_states):
        input_dtype = hidden_states.dtype
        variance = hidden_states.to(torch.float32).pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        if self.use_temporary_parameter:
            weight = self.temp_weight
            bias = self.temp_bias
        else:
            weight = self.weight
            bias = self.bias if hasattr(self, 'bias') else None

        return (weight * hidden_states+bias).to(input_dtype) if bias is not None else (weight * hidden_states).to(input_dtype)

In [7]:
class QuantLlamaMLP(nn.Module):
    def __init__(
        self,
        org_module: nn.Module,
        hidden_size: int,
        intermediate_size: int,
        hidden_act: str,
        args=None,
    ):
        super().__init__()
        # self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        # self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        # self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.gate_proj = QuantLinear(org_module.gate_proj,
                                           args.weight_quant_params,
                                           args.act_quant_params)
        self.down_proj = QuantLinear(org_module.down_proj,
                                           args.weight_quant_params,
                                           args.act_quant_params)
        self.up_proj = QuantLinear(org_module.up_proj,
                                           args.weight_quant_params,
                                           args.act_quant_params)
        self.act_fn = ACT2FN[hidden_act]

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

In [8]:
class QuantLlamaAttention(nn.Module):
    # def __init__(self, fp_attn):
    #     super().__init__()
    #     self.debug = {}
    #     self.hidden_size = fp_attn.config.hidden_size
    #     self.num_heads = fp_attn.config.num_attention_heads
    #     self.head_dim = self.hidden_size // self.num_heads
    #     self.num_kv_heads = fp_attn.config.num_key_value_heads
    #     self.num_groups=self.num_heads//self.num_kv_heads
    #     self.rotary_emb = fp_attn.rotary_fn

    #     # Quantized projections
    #     self.q_proj = QuantLinear(fp_attn.q_proj)
    #     self.k_proj = QuantLinear(fp_attn.k_proj)
    #     self.v_proj = QuantLinear(fp_attn.v_proj)
    #     self.o_proj = QuantLinear(fp_attn.o_proj)

    #     self.qk_matmul = QuantMatMul()
    #     self.pv_matmul = QuantMatMul()

    # def forward(self, hidden_states, attention_mask=None, position_embeddings=None):
    #     B, T, C = hidden_states.shape

    #     # QKV
    #     q = self.q_proj(hidden_states)
    #     k = self.k_proj(hidden_states)
    #     v = self.v_proj(hidden_states)
    #     self.debug["q_q"] = q.detach().cpu()
    #     # reshape → heads
    #     q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    #     k = k.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
    #     v = v.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

    #     # rotary
    #     cos, sin = position_embeddings
    #     q, k = apply_rotary_pos_emb(q, k, cos, sin)

    #     # repeat kv
    #     k = repeat_kv(k, self.num_groups)
    #     v = repeat_kv(v, self.num_groups)

    #     # attention
    #     attn = self.qk_matmul(q, k.transpose(-2, -1))
    #     attn = attn / math.sqrt(self.head_dim)
    #     self.debug["attn_q"] = attn.detach().cpu()
    #     if attention_mask is not None:
    #         attn = attn + attention_mask

    #     attn = torch.softmax(attn, dim=-1)

    #     out = self.pv_matmul(attn, v)
    #     self.debug["out_q"] = out.detach().cpu()
    #     out = out.transpose(1, 2).contiguous().view(B, T, C)

    #     return self.o_proj(out)
    def __init__(self,
                 org_module: nn.Module,
                 config: LlamaConfig,
                 args=None):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.hidden_size // self.num_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.num_key_value_groups = self.num_heads // self.num_key_value_heads
        self.max_position_embeddings = config.max_position_embeddings

        if (self.head_dim * self.num_heads) != self.hidden_size:
            raise ValueError(
                f"hidden_size must be divisible by num_heads (got `hidden_size`: {self.hidden_size}"
                f" and `num_heads`: {self.num_heads})."
            )

        self.rotary_fn = copy.deepcopy(org_module.rotary_fn)
        print(config)
        self.k_proj = QuantLinear(
            org_module.k_proj,
            args.weight_quant_params,
            args.act_quant_params,
        )
        self.v_proj = QuantLinear(
            org_module.v_proj,
            args.weight_quant_params,
            args.act_quant_params,
        )
        self.q_proj = QuantLinear(
            org_module.q_proj,
            args.weight_quant_params,
            args.act_quant_params,
        )
        self.o_proj = QuantLinear(
            org_module.o_proj, args.weight_quant_params, args.act_quant_params
        )
        self.qkt_matmul = QuantMatMul(
            args.q_quant_params, args.k_quant_params, matmul_func=torch.matmul
        )
        self.pv_matmul = QuantMatMul(
            args.p_quant_params, args.v_quant_params, matmul_func=torch.matmul
        )

        self.use_weight_quant = False
        self.use_act_quant = False

    def _shape(self, tensor: torch.Tensor, seq_len: int, bsz: int):
        return tensor.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2).contiguous()

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value: Optional[Tuple[torch.Tensor]] = None,
        output_attentions: bool = False,
        use_cache: bool = False,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:
        bsz, q_len, _ = hidden_states.size()

        # query_states = self.q_proj(hidden_states)
        # key_states = self.k_proj(hidden_states)
        # value_states = self.v_proj(hidden_states)
        query_states = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states =self.k_proj(hidden_states).view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        kv_seq_len = key_states.shape[-2]
        if past_key_value is not None:
            kv_seq_len += past_key_value[0].shape[-2]


        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_value is not None:
            # reuse k, v, self_attention
            key_states = torch.cat([past_key_value[0], key_states], dim=2)
            value_states = torch.cat([past_key_value[1], value_states], dim=2)

        past_key_value = (key_states, value_states) if use_cache else None

        # repeat k/v heads if n_kv_heads < n_heads
        key_states = repeat_kv(key_states, self.num_key_value_groups)
        value_states = repeat_kv(value_states, self.num_key_value_groups)

        query_states = self.qkt_matmul.quant_x1(query_states)
        key_states = self.qkt_matmul.quant_x2(key_states)
        attn_weights = self.qkt_matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(self.head_dim)

        if attn_weights.size() != (bsz, self.num_heads, q_len, kv_seq_len):
            raise ValueError(
                f"Attention weights should be of size {(bsz, self.num_heads, q_len, kv_seq_len)}, but is"
                f" {attn_weights.size()}"
            )

        if attention_mask is not None:
            if attention_mask.size() != (bsz, 1, q_len, kv_seq_len):
                raise ValueError(
                    f"Attention mask should be of size {(bsz, 1, q_len, kv_seq_len)}, but is {attention_mask.size()}"
                )
            attn_weights = attn_weights + attention_mask
            attn_weights = torch.max(attn_weights, torch.tensor(torch.finfo(attn_weights.dtype).min))

        # upcast attention to fp16
        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float16).to(query_states.dtype)
        attn_weights = self.pv_matmul.quant_x1(attn_weights)
        value_states = self.pv_matmul.quant_x2(value_states)
        attn_output = self.pv_matmul(attn_weights, value_states)

        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):
            raise ValueError(
                f"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is"
                f" {attn_output.size()}"
            )

        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.reshape(bsz, q_len, self.hidden_size)

        attn_output = self.o_proj(attn_output)

        if not output_attentions:
            attn_weights = None

        return attn_output, attn_weights, past_key_value

    def set_quant_state(self, weight_quant: bool = False, act_quant: bool = False):
        # setting weight quantization here does not affect actual forward pass
        self.use_weight_quant = weight_quant
        self.use_act_quant = act_quant
        for m in self.modules():
            if isinstance(m, (QuantLinear, QuantMatMul)):
                m.set_quant_state(weight_quant, act_quant)



In [9]:
from transformers.models.llama.modeling_llama import LlamaRMSNorm
pairs = {
            "q_proj":"qkv",
            "o_proj":"out",
            "up_proj":"fc1"
        }
class QuantLlamaDecoderLayer(nn.Module):
    def __init__(self,
                 config: LlamaConfig,
                 ori_layer,
                 args):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.self_attn = QuantLlamaAttention(
            org_module=ori_layer.self_attn,
            config=config,
            args=args,
            )
        self.mlp = QuantLlamaMLP(
            org_module=ori_layer.mlp,
            hidden_size=self.hidden_size,
            intermediate_size=config.intermediate_size,
            hidden_act=config.hidden_act,
            args=args,
        )
        self.input_layernorm = QuantLlamaRMSNorm(
            ori_layer.input_layernorm,
            eps=ori_layer.input_layernorm.variance_epsilon
        )


        self.post_attention_layernorm = QuantLlamaRMSNorm(
            ori_layer.post_attention_layernorm,
            eps=ori_layer.post_attention_layernorm.variance_epsilon
        )
        # self.post_attention_layernorm.weight = nn.Parameter(ori_layer.post_attention_layernorm.weight.clone())

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_value: Optional[Tuple[torch.Tensor]] = None,
        output_attentions: Optional[bool] = False,
        past_key_values=None,
        use_cache: Optional[bool] = False,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs

    ) -> Tuple[torch.FloatTensor, Optional[Tuple[torch.FloatTensor, torch.FloatTensor]]]:
        """
        Args:
            hidden_states (`torch.FloatTensor`): input to the layer of shape `(batch, seq_len, embed_dim)`
            attention_mask (`torch.FloatTensor`, *optional*): attention mask of size
                `(batch, 1, tgt_len, src_len)` where padding elements are indicated by very large negative values.
            output_attentions (`bool`, *optional*):
                Whether or not to return the attentions tensors of all attention layers. See `attentions` under
                returned tensors for more detail.
            use_cache (`bool`, *optional*):
                If set to `True`, `past_key_values` key value states are returned and can be used to speed up decoding
                (see `past_key_values`).
            past_key_value (`Tuple(torch.FloatTensor)`, *optional*): cached past key and value projection states
        """
        residual = hidden_states

        hidden_states = self.input_layernorm(hidden_states)


        # Self Attention
        hidden_states, self_attn_weights, present_key_value = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_value=past_key_value,
            output_attentions=output_attentions,
            use_cache=use_cache,
            position_embeddings=position_embeddings,
        )
        hidden_states = residual + hidden_states

        # Fully Connected
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)


        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states

        outputs = (hidden_states,)

        if output_attentions:
            outputs += (self_attn_weights,)

        if use_cache:
            outputs += (present_key_value,)

        return outputs
    def set_quant_state(self, weight_quant: bool = False, act_quant: bool = False):
          # setting weight quantization here does not affect actual forward pass
          self.use_weight_quant = weight_quant
          self.use_act_quant = act_quant

          for name, m in self.named_modules():
              if isinstance(m, (QuantLinear, QuantMatMul)):
                  m.set_quant_state(weight_quant, act_quant)
    def smooth_and_quant_temporary(self):
        if self.let:
            with torch.no_grad():
                for name, module in self.named_parameters():
                    if "smooth_scale" in name:
                        module.data = truncate_number(module)
            smooth_ln_fcs_temporary(self.input_layernorm,[self.self_attn.q_proj, self.self_attn.k_proj, self.self_attn.v_proj],
                                     self.qkv_smooth_scale,self.qkv_smooth_shift) #4096
            smooth_ln_fcs_temporary(self.post_attention_layernorm,[self.mlp.up_proj,self.mlp.gate_proj],
                                    self.fc1_smooth_scale,self.fc1_smooth_shift) #4096
            smooth_fc_fc_temporary(self.self_attn.v_proj,self.self_attn.o_proj,
                                 self.out_smooth_scale, self.out_smooth_shift) #4096*4096
            smooth_q_k_temporary(self.self_attn.q_proj, self.self_attn.k_proj,
                                 self.qkt_smooth_scale) #4096*4096
            self.mlp.down_proj.temp_weight = self.mlp.down_proj.weight
        else:
            for name, module in self.named_modules():
                if isinstance(module, QuantLinear):
                    module.temp_weight = module.weight
        # quant
        for name, module in self.named_modules():
            if isinstance(module, QuantLinear):
                if hasattr(module, "temp_weight"):
                    name_tmp = name.replace(".","_")
                    if hasattr(self, f"{name_tmp}_smooth_rotate"):
                        r = getattr(self, f"{name_tmp}_smooth_rotate")
                        module.temp_weight.data = module.weight_quantizer(module.temp_weight, rotate=r).data
                    else:
                        module.temp_weight.data = module.weight_quantizer(module.temp_weight).detach().data
                else:
                    name_tmp = name.replace(".","_")
                    if hasattr(self, f"{name_tmp}_smooth_rotate"):
                        r = getattr(self, f"{name_tmp}_smooth_rotate")
                        module.weight.data = module.weight_quantizer(module.weight, rotate=r).data
                    else:
                        module.weight.data = module.weight_quantizer(module.weight).data
                if not hasattr(module, "temp_bias"):
                    module.temp_bias = module.bias
                module.use_temporary_parameter=True

    def clear_temp_variable(self):
       for name, module in self.named_modules():
            if isinstance(module, QuantLinear):
                del module.temp_weight
                del module.temp_bias

    @torch.no_grad()
    def smooth_and_quant_inplace(self):
        if self.let:
            for name, module in self.named_parameters():
                if "smooth_scale" in name:
                    module.data = truncate_number(module)
            smooth_ln_fcs_inplace(self.input_layernorm,[self.self_attn.q_proj, self.self_attn.k_proj, self.self_attn.v_proj],
                                     self.qkv_smooth_scale,self.qkv_smooth_shift)
            smooth_ln_fcs_inplace(self.post_attention_layernorm,[self.mlp.up_proj,self.mlp.gate_proj],
                                    self.fc1_smooth_scale,self.fc1_smooth_shift)
            smooth_fc_fc_inplace(self.self_attn.v_proj,self.self_attn.o_proj,
                                self.out_smooth_scale,self.out_smooth_shift )
            smooth_q_k_inplace(self.self_attn.q_proj, self.self_attn.k_proj,
                                 self.qkt_smooth_scale)
        for name, module in self.named_modules():
            if isinstance(module, QuantLinear):
                name_tmp = name.replace(".","_")
                if hasattr(self, f"{name_tmp}_smooth_rotate"):
                    r = getattr(self, f"{name_tmp}_smooth_rotate")
                    module.weight.data = module.weight_quantizer(module.weight, rotate=r).data
                else:
                    module.weight.data = module.weight_quantizer(module.weight).data
                module.use_temporary_parameter=False

    def let_parameters(self, use_shift=True):
        params = []
        template = "smooth" if use_shift else "smooth_scale"
        for n, m in self.named_parameters():
            if n.find(template) > -1:
                params.append(m)
        return iter(params)

    def lwc_parameters(self):
        params = []
        for n, m in self.named_parameters():
            if n.find('bound_factor') > -1:
                params.append(m)
        return iter(params)

    def rlq_parameters(self, use_shift=True):
        params = []
        template = "smooth" if use_shift else "smooth_scale"
        for n, m in self.named_parameters():
            if n.find('bound_factor') > -1 or n.find(template) > -1:
                params.append(m)
        return iter(params)

    def rlq_state_dict(self, destination=None, prefix='', keep_vars=False):
        if destination is None:
            destination = OrderedDict()
        for name, param in self.named_parameters():
            if name.find('smooth') > -1 or name.find('bound_factor') > -1:
                destination[prefix + name] = param if keep_vars else param.detach()
        return destination


    def rlq_state_dict(self, destination=None, prefix='', keep_vars=False):
        if destination is None:
            destination = OrderedDict()
        for name, param in self.named_parameters():
            if name.find('smooth') > -1 or name.find('bound_factor') > -1:
                destination[prefix + name] = param if keep_vars else param.detach()
        return destination


    def register_scales_and_zeros(self):
        for name, module in self.named_modules():
            if isinstance(module, QuantLinear):
                module.weight_quantizer.register_scales_and_zeros()

### Loading our quantized model and replacing with the layers and scaling factors

In [1]:
from transformers import LlamaForCausalLM, LlamaConfig
model_path='../save_dir'

model = LlamaForCausalLM.from_pretrained(model_path)



c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0318 16:39:36.379000 6616 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2032.95it/s]
LlamaForCausalLM LOAD REPORT from: ../save_dir
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.layers.{0...31}.self_attn.v_proj.weight_quantizer.scales | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.o_proj.bias                    | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.v_proj.bias                    | UNEXPECTED |  | 
model.layers.{0...31}.mlp_up_proj_s

In [9]:
import torch
model.load_state_dict(torch.load("../save_dir/current_model.pth"),strict=False)

_IncompatibleKeys(missing_keys=[], unexpected_keys=['model.layers.0.qkt_smooth_scale', 'model.layers.0.qkv_smooth_shift', 'model.layers.0.qkv_smooth_scale', 'model.layers.0.self_attn_q_proj_smooth_rotate', 'model.layers.0.out_smooth_shift', 'model.layers.0.out_smooth_scale', 'model.layers.0.self_attn_o_proj_smooth_rotate', 'model.layers.0.fc1_smooth_shift', 'model.layers.0.fc1_smooth_scale', 'model.layers.0.mlp_up_proj_smooth_rotate', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.q_proj.weight_quantizer.upbound_factor', 'model.layers.0.self_attn.q_proj.weight_quantizer.lowbound_factor', 'model.layers.0.self_attn.q_proj.weight_quantizer.scales', 'model.layers.0.self_attn.q_proj.weight_quantizer.zeros', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.k_proj.weight_quantizer.upbound_factor', 'model.layers.0.self_attn.k_proj.weight_quantizer.lowbound_factor', 'model.layers.0.self_attn.k_proj.weight_quantizer.scales', 'model.layers.0.self_attn.k_proj.we

In [16]:
model.model.layers[0].self_attn.q_proj.weight

Parameter containing:
tensor([[-0.0057, -0.0126, -0.0011,  ...,  0.0034,  0.0011, -0.0034],
        [ 0.0152, -0.0030,  0.0030,  ..., -0.0061, -0.0091,  0.0061],
        [-0.0129,  0.0086,  0.0000,  ...,  0.0043,  0.0129, -0.0043],
        ...,
        [ 0.0012,  0.0095,  0.0000,  ...,  0.0071, -0.0213,  0.0083],
        [ 0.0241,  0.0093,  0.0019,  ..., -0.0259, -0.0111, -0.0093],
        [-0.0139, -0.0052,  0.0017,  ...,  0.0139,  0.0121, -0.0069]],
       dtype=torch.float16, requires_grad=True)

In [23]:
model.model.layers[16].self_attn.o_proj.weight

Parameter containing:
tensor([[ 1.9300e-02, -3.1371e-02, -2.1686e-03,  ..., -1.0504e-02,
          1.5477e-02, -6.2570e-02],
        [ 2.8690e-03, -2.2615e-02,  2.9639e-02,  ..., -1.8632e-02,
         -2.2221e-02,  1.3834e-02],
        [-1.8876e-02,  3.1827e-03, -6.3524e-04,  ...,  1.0207e-02,
          9.2836e-05,  1.6299e-02],
        ...,
        [ 1.9957e-03,  1.1319e-02,  1.3296e-02,  ..., -5.8113e-03,
          7.1553e-03, -1.0123e-02],
        [-6.2113e-03, -6.3106e-03,  9.7351e-03,  ..., -2.2372e-02,
         -1.3427e-02,  3.1846e-03],
        [-1.8664e-03, -1.3142e-02, -5.3772e-03,  ...,  2.2869e-02,
         -3.9906e-03,  5.4276e-03]], requires_grad=True)

In [ ]:
# 2. Re-apply your layer replacement (Crucial!)
# Your model MUST have QuantLlamaDecoderLayer installed before loading the keys
for i in range(len(model.model.layers)):
    # Create the custom layer (use dummy args if necessary, weights will be overwritten)
    old_layer = model.model.layers[i]
    model.model.layers[i] = QuantLlamaDecoderLayer(config, old_layer,args )

In [3]:
import torch
state_dict = torch.load("../save_improved/current_model.pth", map_location="cpu")


In [5]:
state_dict

OrderedDict([('model.embed_tokens.weight',
              tensor([[ 1.2517e-06, -1.7881e-06, -4.3511e-06,  ...,  8.9407e-07,
                       -6.5565e-06,  8.9407e-07],
                      [ 1.8616e-03, -3.3722e-03,  3.9864e-04,  ..., -8.3008e-03,
                        2.5787e-03, -3.9368e-03],
                      [ 1.0986e-02,  9.8877e-03, -5.0964e-03,  ...,  2.5177e-03,
                        7.7057e-04, -5.0049e-03],
                      ...,
                      [-1.3977e-02, -2.7313e-03, -1.9897e-02,  ..., -1.0437e-02,
                        9.5825e-03, -1.8005e-03],
                      [-1.0742e-02,  9.3384e-03,  1.2939e-02,  ..., -3.3203e-02,
                       -1.6357e-02,  3.3875e-03],
                      [-8.3008e-03, -4.0588e-03, -1.1063e-03,  ...,  3.4790e-03,
                       -1.2939e-02,  3.1948e-05]], dtype=torch.float16)),
             ('model.layers.0.qkt_smooth_scale',
              tensor([1.1406, 1.2598, 0.7783,  ..., 1.7090, 2.6562, 2.1

In [15]:
weight=state_dict['model.layers.17.mlp.up_proj.weight']
scale=state_dict['model.layers.17.mlp.up_proj.weight_quantizer.scales']

In [16]:
scale

tensor([[0.0005],
        [0.0004],
        [0.0004],
        ...,
        [0.0003],
        [0.0005],
        [0.0005]], dtype=torch.float16)

In [17]:
torch.unique(weight[0]*scale[0]).size(0)

686

In [28]:
weight.shape

torch.Size([11008, 4096])